# Prediction of Cardiovascular Diseases using Machine Learning

**BE (Electronics and Communication Engineering) final-year project, Muffakham Jah College of Engineering and Technology (Osmania University), 2021 to 2022**

Abdul Samad Khan, with Shaik Sahera Begum and Mohammed Asif Ahmed. Project guide: Dr. M. A. Raheem.

> **About this notebook.** The original `.ipynb` from 2022 was not kept. This notebook was rebuilt line by line from the code screenshots printed in the thesis (Chapters 5 to 7), so the code matches what was run at the time. Outputs are not stored here; the results recorded in the thesis are noted in markdown under each step. Two small changes were made so it runs on any machine: the dataset path is relative (`data/heart.csv`) instead of a local Windows path, and the `random_state` of the train/test split, which is cut off in the screenshot, is left at the library default. Your numbers may therefore differ slightly from the thesis.

Dataset: the Kaggle heart disease dataset (303 patients, 13 clinical attributes plus `target`). Download it from Kaggle and save it as `data/heart.csv`.

## 1. Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import sklearn.metrics as sm
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

## 2. Load and check the data

In [ ]:
cvd = pd.read_csv("data/heart.csv")
cvd.head()

In [ ]:
print('The dimensions are', cvd.shape)

Thesis result: `The dimensions are (303, 14)`.

In [ ]:
cvd.isnull().any()

Thesis result: no missing values in any column.

## 3. Exploratory data analysis

### Counts by attribute

In [ ]:
print(cvd.groupby('age').size())

In [ ]:
fig=plt.figure(figsize=(5,5))
sns.countplot(cvd['age'],label="count")

In [ ]:
print(cvd.groupby('sex').size())

In [ ]:
sns.countplot(cvd['sex'],label="count")

In [ ]:
print(cvd.groupby('cp').size())

In [ ]:
sns.countplot(cvd['cp'],label="count")

In [ ]:
print(cvd.groupby('trestbps').size())

In [ ]:
fig=plt.figure(figsize=(20,10))
sns.countplot(cvd['trestbps'],label="count")

In [ ]:
print(cvd.groupby('chol').size())

In [ ]:
fig=plt.figure(figsize=(20,10))
sns.countplot(cvd['chol'],label="count")

Thesis results: 207 male (`sex = 1`) and 96 female (`sex = 0`); chest pain type 0 is the most common (143 patients); the most common ages are 57 to 59.

### Histograms of all attributes

In [ ]:
fig=plt.figure(figsize=(15,20))
ax=fig.gca()
cvd.hist(ax=ax)

### Correlation heatmap

In [ ]:
corrmat = cvd.corr()
f, ax = plt.subplots(figsize =(15, 15))
sns.heatmap(corrmat, ax = ax, cmap ="YlGnBu", linewidths = 0.1)

### Scatter plots against the target

In [ ]:
a = cvd["age"]
b = cvd["target"]
plt.scatter(a,b,color='red')
plt.title('age vs target')
plt.xlabel('age')
plt.ylabel('target')
plt.show()

In [ ]:
a = cvd["sex"]
b = cvd["target"]
plt.scatter(a,b,color='red')
plt.title('sex vs target')
plt.xlabel('sex')
plt.ylabel('target')
plt.show()

In [ ]:
a = cvd["cp"]
b = cvd["target"]
plt.scatter(a,b,color='red')
plt.title('cp vs target')
plt.xlabel('cp')
plt.ylabel('target')
plt.show()

In [ ]:
a = cvd["trestbps"]
b = cvd["target"]
plt.scatter(a,b,color='red')
plt.title('trestbps vs target')
plt.xlabel('trestbps')
plt.ylabel('target')
plt.show()

### Box plots (outlier check)

In [ ]:
plt.boxplot(cvd["age"])

In [ ]:
plt.boxplot(cvd["sex"])

In [ ]:
plt.boxplot(cvd["cp"])

In [ ]:
plt.boxplot(cvd["trestbps"])

Thesis observation: `trestbps` (resting blood pressure) shows a few high outliers above 170 mm Hg.

## 4. Train/test split

A stratified split keeps the same share of patients with and without disease in both sets. With the default 25% test size this gives 227 training rows and 76 test rows, which matches the thesis.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(cvd.loc[:, cvd.columns != 'target'], cvd['target'], stratify=cvd['target'])

## 5. K-Nearest Neighbours

10-fold cross-validation on the full data for k = 1 to 20.

In [ ]:
y = cvd['target']
X = cvd.drop(['target'], axis = 1)

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
knn_scores = []
for k in range(1,21):
    knn_classifier = KNeighborsClassifier(n_neighbors = k)
    score=cross_val_score(knn_classifier,X,y,cv=10)
    knn_scores.append(score.mean())

In [ ]:
fig=plt.figure(figsize=(20,10))
plt.plot([k for k in range(1, 21)], knn_scores, color = 'red')
for i in range(1,21):
    plt.text(i, knn_scores[i-1], (i, knn_scores[i-1]))
plt.xticks([i for i in range(1, 21)])
plt.xlabel('Number of Neighbors (K)')
plt.ylabel('Scores')
plt.title('K Neighbors Classifier scores for different K values')

In [ ]:
knn_classifier = KNeighborsClassifier(n_neighbors = 7)
score=cross_val_score(knn_classifier,X,y,cv=10)

In [ ]:
score.mean()

Thesis result: mean 10-fold accuracy **0.673** with k = 7.

## 6. Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
logreg = LogisticRegression().fit(X_train, y_train)
print("Training set score: {:.3f}".format(logreg.score(X_train, y_train)))
print("Test set score: {:.3f}".format(logreg.score(X_test, y_test)))

Thesis result: training 0.859, test **0.829**.

In [ ]:
logreg001 = LogisticRegression(C=0.01).fit(X_train, y_train)
print("Training set accuracy: {:.3f}".format(logreg001.score(X_train, y_train)))
print("Test set accuracy: {:.3f}".format(logreg001.score(X_test, y_test)))

Thesis result (C = 0.01): training 0.780, test 0.724.

In [ ]:
logreg100 = LogisticRegression(C=100).fit(X_train, y_train)
print("Training set accuracy: {:.3f}".format(logreg100.score(X_train, y_train)))
print("Test set accuracy: {:.3f}".format(logreg100.score(X_test, y_test)))

Thesis result (C = 100): training 0.859, test 0.829.

## 7. Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
tree = DecisionTreeClassifier(random_state=0)
tree.fit(X_train, y_train)
print("Accuracy on training set: {:.3f}".format(tree.score(X_train, y_train)))
print("Accuracy on test set: {:.3f}".format(tree.score(X_test, y_test)))

Thesis result (no depth limit): training 1.000, test 0.789. The unpruned tree overfits.

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=0)
tree.fit(X_train, y_train)
print("Accuracy on training set: {:.3f}".format(tree.score(X_train, y_train)))
print("Accuracy on test set: {:.3f}".format(tree.score(X_test, y_test)))

Thesis result (max_depth = 3): training 0.850, test **0.816**.

In [ ]:
print("Feature importances:\n{}".format(tree.feature_importances_))

In [ ]:
cvd_features = [x for i,x in enumerate(cvd.columns) if i!=13]
def plot_feature_importances_cvd(model):
    plt.figure(figsize=(8,6))
    n_features = 13
    plt.barh(range(n_features), model.feature_importances_, align='center')
    plt.yticks(np.arange(n_features), cvd_features)
    plt.xlabel("Feature importance")
    plt.ylabel("Feature")
    plt.ylim(-1, n_features)
plot_feature_importances_cvd(tree)
plt.savefig('feature_importance')

Thesis result: the depth-3 tree relies mainly on `cp` (0.52), `thal` (0.21) and `ca` (0.18).

## 8. Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, random_state=0)
rf.fit(X_train, y_train)
print("Accuracy on training set: {:.3f}".format(rf.score(X_train, y_train)))
print("Accuracy on test set: {:.3f}".format(rf.score(X_test, y_test)))

Thesis result: training 1.000, test 0.829.

In [ ]:
rf1 = RandomForestClassifier(max_depth=3, n_estimators=100, random_state=0)
rf1.fit(X_train, y_train)
print("Accuracy on training set: {:.3f}".format(rf1.score(X_train, y_train)))
print("Accuracy on test set: {:.3f}".format(rf1.score(X_test, y_test)))

Thesis result (max_depth = 3): training 0.916, test 0.816.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
classifier.fit(X_train, y_train)
print("Accuracy on training set: {:.3f}".format(classifier.score(X_train, y_train)))
print("Accuracy on test set: {:.3f}".format(classifier.score(X_test, y_test)))

Thesis result (10 trees, entropy): training 0.987, test **0.829**. This is the model used for the real-time prediction below.

In [ ]:
X_test

In [ ]:
y_pred = classifier.predict(X_test)

In [ ]:
y_pred

### ROC curve

In [ ]:
import sklearn.metrics as metrics
fpr, tpr, threshold = metrics.roc_curve(y_test, y_pred)
roc_auc = metrics.auc(fpr, tpr)

In [ ]:
plt.title('Receiver Operating Characteristic')
plt.plot(fpr, tpr, 'b', label = 'AUC = %0.2f' % roc_auc)
plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.show()

Thesis result: **AUC = 0.82**. The curve is built from hard 0/1 predictions, so it has a single bend; using `classifier.predict_proba(X_test)[:, 1]` would give a smoother curve.

In [ ]:
plot_feature_importances_cvd(rf)

## 9. Support Vector Machine

In [ ]:
from sklearn.svm import SVC
svc = SVC()
svc.fit(X_train, y_train)
print("Accuracy on training set: {:.2f}".format(svc.score(X_train, y_train)))
print("Accuracy on test set: {:.2f}".format(svc.score(X_test, y_test)))

Thesis result (unscaled): training 0.66, test 0.58. SVM is sensitive to feature scale, so the data is rescaled next.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)
svc = SVC()
svc.fit(X_train_scaled, y_train)
print("Accuracy on training set: {:.2f}".format(svc.score(X_train_scaled, y_train)))
print("Accuracy on test set: {:.2f}".format(svc.score(X_test_scaled, y_test)))

Thesis result (MinMax scaled): training 0.88, test **0.75**.

## 10. Real-time prediction of CVD

The 13 attributes of a patient are entered in column order and the Random Forest classifier returns 1 (CVD) or 0 (no CVD).

In [ ]:
new_input = [[55,0,1,135,250,0,0,161,0,1.4,1,0,2]]
# get prediction for new input
new_output = classifier.predict(new_input)
# summarize input and output
print(new_output);
if (new_output == 0):
 print("THE PATIENT DOES NOT HAVE 'CARDIOVASCULAR DISEASES'")
else :
 print("THE PATIENT HAS 'CARDIOVASCULAR DISEASES'")

Thesis result: `[1]` THE PATIENT HAS 'CARDIOVASCULAR DISEASES'.

## 11. Summary of results (Table 7.1 of the thesis)

| Model | Accuracy |
|---|---|
| K-Nearest Neighbours (k = 7, 10-fold CV) | 67% |
| Logistic Regression | 82% |
| Random Forest | 83.9% |
| Decision Tree (max_depth = 3) | 81.6% |
| Support Vector Machine (MinMax scaled) | 75% |

Table 7.1 gives 83.9% for Random Forest, while the notebook output printed in Chapter 6 shows 0.829 on the test set; both values are reported as they appear in the thesis. Random Forest was selected for real-time prediction and for the IBM Watson Studio deployment.